# Train NOVA on an open-weight language model

This notebook fine-tunes an **Apache-2.0 Qwen3-0.6B** model with a small LoRA adapter using NOVA's manually curated examples. It follows a real **supervised fine-tuning** step, but **does not** reproduce OpenAI or Anthropic private data or their frontier-scale training.

**Before Run All:** choose **Runtime → Change runtime type → T4 GPU** (if a GPU is available). Colab free GPU access is limited and not guaranteed. The script never pushes weights or training messages to a model hub or GitHub. Only use examples you have rights and consent to train on.

Official model: https://huggingface.co/Qwen/Qwen3-0.6B


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This experiment needs a GPU. Choose Runtime → Change runtime type → GPU. Free GPUs may be unavailable.')
!pip -q install -U 'transformers>=4.55' 'trl>=0.21' 'peft>=0.17' 'datasets>=3.6' 'accelerate>=1.5' 'safetensors>=0.4'


In [ ]:
import os
if not os.path.exists('/content/NOVA-AI/.git'):
    !git clone --depth 1 https://github.com/Exclusive2knight/NOVA-AI.git /content/NOVA-AI
%cd /content/NOVA-AI
!python training/build_nova_sft.py --input training/nova_sft_examples.jsonl --out-dir nova-data/sft


In [ ]:
# One real supervised fine-tuning run. Start small to avoid overfitting.
MAX_TRAINING_STEPS = 30
!python training/finetune_open_model.py --data-dir nova-data/sft --out-dir nova-data/nova-qwen-lora --max-steps {MAX_TRAINING_STEPS}


## Evaluate after training

The test examples are held out during training. Use this test **once** to compare the pretrained model with the LoRA-adapted model. A lower loss on eight small, in-domain examples does **not** demonstrate general intelligence or parity with ChatGPT or Claude. Do not adjust training based on test results.


In [ ]:
!python training/evaluate_nova_open_model.py --data-dir nova-data/sft --adapter nova-data/nova-qwen-lora/adapter --output nova-data/nova-qwen-lora/test_report.json


In [ ]:
# See what NOVA says with your newly trained adapter (on this GPU only).\n!python training/chat_nova_open_model.py --adapter nova-data/nova-qwen-lora/adapter --prompt 'How are we doing with NOVA?'\n

In [ ]:
# Keep this downloaded adapter private; there is no automatic website deployment.
import shutil
from google.colab import files
name = shutil.make_archive('/content/NOVA_Qwen3_LoRA_training_results', 'zip', 'nova-data/nova-qwen-lora')
print('Created adapter/report archive:', name)
files.download(name)
